# 01 — Bona Data Analysis
## SunnyBest Retail Forecasting System — Dataset Exploration

This notebook analyses all available datasets in the SunnyBest SFS project:
- `sunnybest_merged_df.csv` — Master merged dataset (daily grain)
- `weekly_sales_v4_promotions.csv` — Latest weekly model input (v4)
- `weekly_sales_v3_calendar.csv` — Weekly v3 with calendar features
- `weekly_sales_v2.csv` — Weekly v2 baseline
- `weekly_sales.csv` — Weekly v1 (first iteration)
- `elasticity_by_category.csv` — Price elasticity per category
- `weekly_forecasts.csv` — Model predictions
- `weekly_actuals.csv` — Ground truth for monitoring

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

BASE = "../data/processed"
OUT  = "../data/outputs"

# Load all datasets
df       = pd.read_csv(f"{BASE}/sunnybest_merged_df.csv", parse_dates=["date"])
wk_v4    = pd.read_csv(f"{BASE}/weekly_sales_v4_promotions.csv")
wk_v3    = pd.read_csv(f"{BASE}/weekly_sales_v3_calendar.csv")
wk_v2    = pd.read_csv(f"{BASE}/weekly_sales_v2.csv")
wk_v1    = pd.read_csv(f"{BASE}/weekly_sales.csv")
elast    = pd.read_csv(f"{BASE}/elasticity_by_category.csv")
forecasts = pd.read_csv(f"{OUT}/weekly_forecasts.csv", parse_dates=["week_start"])
actuals   = pd.read_csv(f"{OUT}/weekly_actuals.csv", parse_dates=["week_start"])

print("All datasets loaded successfully.")

---
## 1. Dataset Overview — Shape, Columns & Missing Values

In [ ]:
datasets = {
    "merged_df (daily)":        df,
    "weekly_v4_promotions":     wk_v4,
    "weekly_v3_calendar":       wk_v3,
    "weekly_v2":                wk_v2,
    "weekly_v1":                wk_v1,
    "elasticity_by_category":   elast,
    "weekly_forecasts":         forecasts,
    "weekly_actuals":           actuals,
}

summary_rows = []
for name, d in datasets.items():
    missing_pct = (d.isnull().sum().sum() / d.size * 100).round(2)
    summary_rows.append({
        "Dataset":       name,
        "Rows":          f"{len(d):,}",
        "Columns":       d.shape[1],
        "Missing %":     f"{missing_pct}%",
        "Date Range":    f"{d.iloc[:,0].min()} → {d.iloc[:,0].max()}" if "date" in d.columns or "week_start" in d.columns else "—",
    })

pd.DataFrame(summary_rows).set_index("Dataset")

In [ ]:
# Column-level missing values for the master dataset
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if missing.empty:
    print("No missing values in the master merged dataset.")
else:
    print("Columns with missing values:")
    display(missing.to_frame("missing_count").assign(pct=lambda x: (x/len(df)*100).round(2)))

---
## 2. Master Dataset — Key Statistics & Distributions

In [ ]:
print(f"Date range : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Stores     : {df['store_id'].nunique()} unique stores")
print(f"Products   : {df['product_id'].nunique()} unique products")
print(f"Categories : {df['category'].nunique()} — {list(df['category'].unique())}")
print(f"Cities     : {df['city'].nunique()} — {list(df['city'].unique())}")
print(f"Total rows : {len(df):,}")
print()
df[["units_sold", "revenue", "price", "discount_pct", "starting_inventory"]].describe().round(2)

In [ ]:
# Distribution of units_sold and revenue
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df["units_sold"].clip(upper=df["units_sold"].quantile(0.99)), bins=50, color="#4C72B0", edgecolor="white")
axes[0].set_title("Distribution of Units Sold (daily, 99th pct clip)")
axes[0].set_xlabel("units_sold")

axes[1].hist(df["revenue"].clip(upper=df["revenue"].quantile(0.99)) / 1e6, bins=50, color="#DD8452", edgecolor="white")
axes[1].set_title("Distribution of Revenue (daily, ₦M)")
axes[1].set_xlabel("revenue (₦M)")

plt.tight_layout()
plt.show()

---
## 3. Sales Analysis — By Store, Category & Time

In [ ]:
# Total revenue by store
store_rev = df.groupby("store_name")["revenue"].sum().sort_values(ascending=True) / 1e9

fig, ax = plt.subplots(figsize=(12, 5))
store_rev.plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Total Revenue by Store (₦B)")
ax.set_xlabel("Revenue (₦ Billions)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"₦{x:.1f}B"))
plt.tight_layout()
plt.show()

In [ ]:
# Units sold & revenue by category
cat_stats = df.groupby("category").agg(
    total_units=("units_sold", "sum"),
    total_revenue=("revenue", "sum"),
    avg_price=("price", "mean"),
    stockout_rate=("stockout_occurred", "mean"),
).sort_values("total_revenue", ascending=False).round(2)
cat_stats["total_revenue_M"] = (cat_stats["total_revenue"] / 1e6).round(1)
display(cat_stats[["total_units", "total_revenue_M", "avg_price", "stockout_rate"]].rename(
    columns={"total_revenue_M": "revenue (₦M)"}))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
cat_stats["total_units"].sort_values().plot(kind="barh", ax=axes[0], color="#55A868")
axes[0].set_title("Total Units Sold by Category")

cat_stats["total_revenue_M"].sort_values().plot(kind="barh", ax=axes[1], color="#C44E52")
axes[1].set_title("Total Revenue by Category (₦M)")

plt.tight_layout()
plt.show()

In [ ]:
# Monthly revenue trend over time
monthly = df.groupby(df["date"].dt.to_period("M"))["revenue"].sum() / 1e6
monthly.index = monthly.index.astype(str)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly.index, monthly.values, color="#4C72B0", linewidth=2)
ax.fill_between(monthly.index, monthly.values, alpha=0.15, color="#4C72B0")
ax.set_title("Monthly Revenue Trend (₦M)")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (₦M)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Average units sold by day of week and season
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow = df.groupby("day_of_week")["units_sold"].mean().reindex(dow_order)
dow.plot(kind="bar", ax=axes[0], color="#8172B2", rot=30)
axes[0].set_title("Avg Units Sold by Day of Week")
axes[0].set_ylabel("Avg Units Sold")

season = df.groupby("season")["units_sold"].mean().sort_values(ascending=False)
season.plot(kind="bar", ax=axes[1], color="#64B5CD", rot=0)
axes[1].set_title("Avg Units Sold by Season")

plt.tight_layout()
plt.show()

---
## 4. Inventory & Stockout Analysis

In [ ]:
overall_stockout = df["stockout_occurred"].mean() * 100
print(f"Overall stockout rate: {overall_stockout:.2f}%")

# Stockout rate by category and city
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

so_cat = df.groupby("category")["stockout_occurred"].mean().sort_values(ascending=True) * 100
so_cat.plot(kind="barh", ax=axes[0], color="#C44E52")
axes[0].set_title("Stockout Rate by Category (%)")
axes[0].set_xlabel("Stockout Rate (%)")

so_city = df.groupby("city")["stockout_occurred"].mean().sort_values(ascending=True) * 100
so_city.plot(kind="barh", ax=axes[1], color="#DD8452")
axes[1].set_title("Stockout Rate by City (%)")

plt.tight_layout()
plt.show()

In [ ]:
# Inventory levels over time
inv_monthly = df.groupby(df["date"].dt.to_period("M"))["starting_inventory"].mean()
inv_monthly.index = inv_monthly.index.astype(str)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(inv_monthly.index, inv_monthly.values, color="#55A868", linewidth=2)
ax.fill_between(inv_monthly.index, inv_monthly.values, alpha=0.15, color="#55A868")
ax.set_title("Average Starting Inventory Over Time")
ax.set_xlabel("Month")
ax.set_ylabel("Avg Units in Stock")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

---
## 5. Promotion Analysis

In [ ]:
promo_rate = df["promo_flag"].mean() * 100
print(f"Overall promotion rate: {promo_rate:.2f}% of transaction days")

# Avg units sold: promo vs no promo
promo_lift = df.groupby("promo_flag")["units_sold"].mean()
promo_lift.index = ["No Promo", "Promo"]
print(f"\nAvg units sold (No Promo): {promo_lift['No Promo']:.2f}")
print(f"Avg units sold (Promo):    {promo_lift['Promo']:.2f}")
print(f"Promo lift: {((promo_lift['Promo'] / promo_lift['No Promo']) - 1) * 100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

promo_lift.plot(kind="bar", ax=axes[0], color=["#4C72B0", "#C44E52"], rot=0)
axes[0].set_title("Avg Units Sold: Promo vs No Promo")
axes[0].set_ylabel("Avg Units Sold")

# Promo type breakdown
promo_types = df[df["promo_flag"] == 1]["promo_type"].value_counts()
promo_types.plot(kind="bar", ax=axes[1], color="#8172B2", rot=20)
axes[1].set_title("Promotion Type Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Promo lift by category
promo_by_cat = df.groupby(["category", "promo_flag"])["units_sold"].mean().unstack()
promo_by_cat.columns = ["No Promo", "Promo"]
promo_by_cat["lift_%"] = ((promo_by_cat["Promo"] / promo_by_cat["No Promo"]) - 1) * 100
promo_by_cat = promo_by_cat.sort_values("lift_%", ascending=False)
display(promo_by_cat.round(2))

promo_by_cat["lift_%"].sort_values().plot(kind="barh", color="#55A868", figsize=(10, 4))
plt.title("Promo Uplift % by Category")
plt.xlabel("Uplift (%)")
plt.axvline(0, color="black", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.show()

---
## 6. Weekly Sales Dataset Evolution (v1 → v4)

In [ ]:
# Compare dataset versions: shape and column additions
versions = {"v1": wk_v1, "v2": wk_v2, "v3": wk_v3, "v4": wk_v4}
ver_summary = []
for v, d in versions.items():
    ver_summary.append({"Version": v, "Rows": len(d), "Columns": d.shape[1], "Columns List": list(d.columns)})

ver_df = pd.DataFrame(ver_summary).set_index("Version")
display(ver_df[["Rows", "Columns"]])

# New columns added in each version
prev_cols = set()
for v, row in ver_df.iterrows():
    new_cols = set(row["Columns List"]) - prev_cols
    print(f"\n{v} — new columns added ({len(new_cols)}): {sorted(new_cols)}")
    prev_cols = set(row["Columns List"])

In [ ]:
# Weekly units sold distribution across versions
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
for ax, (v, d), c in zip(axes, versions.items(), colors):
    ax.hist(d["units_sold"].clip(upper=d["units_sold"].quantile(0.99)), bins=40, color=c, edgecolor="white")
    ax.set_title(f"{v}: units_sold\nμ={d['units_sold'].mean():.1f}")
    ax.set_xlabel("units_sold")
plt.suptitle("Weekly Units Sold Distribution Across Dataset Versions", y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Price Elasticity by Category

In [ ]:
display(elast.sort_values("price_elasticity"))

fig, ax = plt.subplots(figsize=(10, 4))
elast_sorted = elast.sort_values("price_elasticity")
colors_e = ["#C44E52" if v < -1 else "#DD8452" if v < 0 else "#55A868" for v in elast_sorted["price_elasticity"]]
ax.barh(elast_sorted["category"], elast_sorted["price_elasticity"], color=colors_e)
ax.axvline(-1, color="black", linewidth=1, linestyle="--", label="Elastic threshold (-1)")
ax.axvline(0, color="grey", linewidth=0.8, linestyle=":")
ax.set_title("Price Elasticity by Category\n(< -1 = elastic, -1 to 0 = inelastic)")
ax.set_xlabel("Price Elasticity Coefficient")
ax.legend()
plt.tight_layout()
plt.show()

print("\nInterpretation:")
for _, row in elast_sorted.iterrows():
    tag = "ELASTIC (price-sensitive)" if row["price_elasticity"] < -1 else "inelastic"
    print(f"  {row['category']:<25} {row['price_elasticity']:>8.4f}  →  {tag}")

---
## 8. Forecast vs Actuals — Model Performance Check

In [ ]:
print(f"Forecasts : {len(forecasts):,} rows | date range: {forecasts['week_start'].min().date()} → {forecasts['week_start'].max().date()}")
print(f"Actuals   : {len(actuals):,} rows  | date range: {actuals['week_start'].min().date()} → {actuals['week_start'].max().date()}")

# Merge on common keys
merged_fa = pd.merge(
    forecasts[["week_start", "store_id", "product_id", "predicted_units"]],
    actuals[["week_start", "store_id", "product_id", "actual_units"]],
    on=["week_start", "store_id", "product_id"],
    how="inner"
)
print(f"\nMatched rows for comparison: {len(merged_fa):,}")

if len(merged_fa) > 0:
    mae  = (merged_fa["predicted_units"] - merged_fa["actual_units"]).abs().mean()
    rmse = np.sqrt(((merged_fa["predicted_units"] - merged_fa["actual_units"]) ** 2).mean())
    mape = ((merged_fa["predicted_units"] - merged_fa["actual_units"]).abs() / merged_fa["actual_units"].replace(0, np.nan)).mean() * 100
    print(f"\nMAE  : {mae:.3f}")
    print(f"RMSE : {rmse:.3f}")
    print(f"MAPE : {mape:.2f}%")
else:
    print("No overlapping dates between forecasts and actuals — forecasts are future-dated.")

In [ ]:
# Aggregate weekly total for visual trend comparison
agg_actuals   = actuals.groupby("week_start")["actual_units"].sum()
agg_forecasts = forecasts.groupby("week_start")["predicted_units"].sum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(agg_actuals.index, agg_actuals.values, label="Actuals", color="#4C72B0", linewidth=2)
ax.plot(agg_forecasts.index, agg_forecasts.values, label="Forecasts", color="#C44E52", linewidth=2, linestyle="--")
ax.set_title("Weekly Total Units — Actuals vs Forecasts")
ax.set_ylabel("Total Units Sold")
ax.set_xlabel("Week")
ax.legend()
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

---
## 9. Key Findings Summary